# Drone Deployment Optimization

This notebook compares three gradient-free optimization methods for drone deployment:
1. **Differential Evolution (DE)** — population-based stochastic search (scipy)
2. **Alternating Optimization** — decomposed position/orientation sub-problems
3. **Particle Swarm Optimization (PSO)** — swarm intelligence with inertia decay and velocity clamping

All start from the same K-means baseline and optimize drone positions and antenna orientations to maximize coverage and critical-user satisfaction.

**Key improvements over initial version:**
- PSO now uses velocity clamping (`v_max = 0.2 × range`) and linear inertia decay (0.9 → 0.4)
- Critical-user penalty increased (λ=50) so the optimizer strongly prioritizes critical SINR satisfaction
- Alternating method uses full-problem cost evaluation for convergence checks
- Altitude optimization can be enabled for additional degrees of freedom

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from dronecomm.config import Config
from dronecomm.optimize import (
    OptimizationProblem,
    ObjectiveFunction,
    build_bounds,
    evaluate_scenario,
    flatten,
    optimize_alternating,
    optimize_de,
    optimize_pso,
    run_comparison,
)
from dronecomm.scenario import create_scenario
from dronecomm.visualization import plot_sinr_heatmap, plot_sinr_cdf
from dronecomm.antenna import ParametricAntenna
from dronecomm.channel import ChannelModel, ENVIRONMENTS
from dronecomm.sinr import dbm_to_watts

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print('Ready.')

## 1. Configuration and Baseline

In [ ]:
config = Config.from_yaml('../configs/optimize.yaml')
SEED = 42

net = config.network
sc_cfg = config.scenario

# Create baseline scenario (K-means placement)
baseline = create_scenario(
    n_drones=net.n_drones,
    placement='kmeans',
    altitude_m=net.altitude_m,
    dl_tilt_deg=net.dl_tilt_deg,
    bh_tilt_deg=net.bh_tilt_deg,
    n_clusters=sc_cfg.n_clusters,
    users_per_cluster_mean=sc_cfg.users_per_cluster_mean,
    cluster_spread_m=sc_cfg.cluster_spread_m,
    area_size_m=sc_cfg.area_size_m,
    critical_fraction=sc_cfg.critical_fraction,
    min_separation_m=net.min_separation_m,
    seed=SEED,
)

# Evaluate baseline with multiple seeds
baseline_metrics = evaluate_scenario(baseline, config, n_seeds=10, base_seed=SEED)
print(f'Baseline ({net.n_drones} drones, {net.altitude_m}m altitude, {sc_cfg.area_size_m}m area)')
print(f'  Coverage:             {baseline_metrics.coverage_fraction:.1%}')
print(f'  Critical satisfied:   {baseline_metrics.critical_sinr_satisfied:.1%}')
print(f'  Throughput:           {baseline_metrics.sum_throughput_mbps:.0f} Mbps')
print(f'  Mean SINR:            {baseline_metrics.mean_sinr_db:.1f} dB')
print(f'  5th pctl SINR:        {baseline_metrics.sinr_5th_percentile_db:.1f} dB')
print(f'  Users: {baseline_metrics.n_users} ({baseline_metrics.n_critical} critical)')

## 2. Run All Three Optimizers

We run DE, Alternating, and PSO from the same K-means baseline. This takes a few minutes.

In [ ]:
results = run_comparison(
    config,
    seed=SEED,
    methods=['baseline', 'de', 'alternating', 'pso'],
    de_maxiter=200,
    pso_maxiter=200,
    pso_n_particles=90,  # >= dimension (90) for more exploration
    alt_outer_iters=10,
    optimize_altitude=False,
    verbose=True,
)

## 3. Summary Comparison Table

In [ ]:
print(f'{"Method":<15} {"Coverage":>9} {"Critical":>9} {"Throughput":>12} '
      f'{"Mean SINR":>10} {"5th pctl":>9} {"Evals":>7} {"Time":>7}')
print('-' * 85)
for name, res in results.items():
    m = res.final_metrics
    print(f'{name:<15} {m.coverage_fraction:>8.1%} '
          f'{m.critical_sinr_satisfied:>8.1%} '
          f'{m.sum_throughput_mbps:>10.0f} Mb '
          f'{m.mean_sinr_db:>9.1f} dB'
          f'{m.sinr_5th_percentile_db:>8.1f} dB'
          f'{res.n_evaluations:>7d} '
          f'{res.elapsed_seconds:>6.1f}s')

## 4. Convergence Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = {'de': 'tab:blue', 'alternating': 'tab:orange', 'pso': 'tab:green'}
labels = {'de': 'Differential Evolution', 'alternating': 'Alternating Opt.', 'pso': 'PSO'}

for method in ['de', 'alternating', 'pso']:
    if method in results:
        hist = results[method].history
        ax.plot(range(len(hist)), hist,
                color=colors[method], label=labels[method], linewidth=1.5)

# Baseline reference
if 'baseline' in results:
    ax.axhline(results['baseline'].best_cost, color='gray', linestyle='--',
               label='Baseline (K-means)', alpha=0.7)

ax.set_xlabel('Objective Evaluations')
ax.set_ylabel('Best Cost (lower is better)')
ax.set_title('Optimization Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. SINR Heat Maps: Baseline vs Best Optimizer

In [ ]:
# Build models for heat map
ant = config.antenna
ch_cfg = config.channel
dl_antenna = ParametricAntenna(
    g_max_dbi=ant.dl_g_max_dbi,
    beamwidth_deg=ant.dl_beamwidth_deg,
    sla_db=ant.dl_sla_db,
)
env = ENVIRONMENTS[ch_cfg.environment]
channel = ChannelModel(env=env, f_c=ch_cfg.carrier_freq_hz)
p_tx_dl_w = dbm_to_watts(net.p_tx_dl_dbm)

# Find best optimizer
opt_methods = [m for m in results if m != 'baseline']
best_method = min(opt_methods, key=lambda m: results[m].best_cost)
best_result = results[best_method]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (label, scenario) in zip(axes, [
    ('Baseline (K-means)', baseline),
    (f'Optimized ({best_method.upper()})', best_result.best_scenario),
]):
    plot_sinr_heatmap(
        drone_positions=scenario.drone_positions,
        dl_tilt_rad=scenario.dl_tilt_rad,
        dl_azimuth_rad=scenario.dl_azimuth_rad,
        dl_antenna=dl_antenna,
        channel=channel,
        p_tx_dl_w=p_tx_dl_w,
        area_size_m=sc_cfg.area_size_m,
        noise_psd_dbm_hz=net.noise_psd_dbm_hz,
        bandwidth_hz=net.bandwidth_hz,
        grid_resolution=80,
        title=f'SINR Heat Map — {label}',
    )
    # Reuse the axes from the generated figure
    plt.close()  # close the individual figure

# Re-draw on side-by-side axes
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax_idx, (label, scenario) in enumerate([
    ('Baseline (K-means)', baseline),
    (f'Optimized ({best_method.upper()})', best_result.best_scenario),
]):
    from dronecomm.interference import downlink_power_matrix
    from dronecomm.sinr import compute_sinr, nearest_drone_association

    xs = np.linspace(0, sc_cfg.area_size_m, 80)
    ys = np.linspace(0, sc_cfg.area_size_m, 80)
    xx, yy = np.meshgrid(xs, ys)
    grid_pos = np.column_stack([xx.ravel(), yy.ravel(), np.zeros(xx.size)])

    power_mat = downlink_power_matrix(
        drone_positions=scenario.drone_positions,
        dl_tilt_rad=scenario.dl_tilt_rad,
        dl_azimuth_rad=scenario.dl_azimuth_rad,
        user_positions=grid_pos,
        dl_antenna=dl_antenna, channel=channel, p_tx_dl_w=p_tx_dl_w,
    )
    assoc = nearest_drone_association(scenario.drone_positions, grid_pos)
    sinr_res = compute_sinr(power_mat, assoc, net.noise_psd_dbm_hz, net.bandwidth_hz)
    sinr_map = sinr_res.sinr_db.reshape(80, 80)

    ax = axes[ax_idx]
    im = ax.pcolormesh(xx, yy, sinr_map, cmap='RdYlGn', shading='auto',
                       vmin=-10, vmax=40)
    fig.colorbar(im, ax=ax, label='SINR [dB]')
    ax.scatter(scenario.drone_positions[:, 0], scenario.drone_positions[:, 1],
               c='black', s=60, marker='^', zorder=5, label='Drones')
    ax.set_xlabel('X [m]')
    ax.set_ylabel('Y [m]')
    ax.set_title(label)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. SINR CDF Comparison

In [ ]:
from dronecomm.optimize import _build_models, _evaluate_single
from dronecomm.sinr import compute_sinr, nearest_drone_association
from dronecomm.interference import downlink_power_matrix

dl_ant, bh_ant, ch = _build_models(config)

fig, ax = plt.subplots(figsize=(9, 5))

plot_colors = {'baseline': 'gray', 'de': 'tab:blue',
               'alternating': 'tab:orange', 'pso': 'tab:green'}
plot_labels = {'baseline': 'Baseline', 'de': 'DE',
               'alternating': 'Alternating', 'pso': 'PSO'}

for method, res in results.items():
    sc = res.best_scenario
    power_mat = downlink_power_matrix(
        sc.drone_positions, sc.dl_tilt_rad, sc.dl_azimuth_rad,
        sc.user_positions, dl_ant, ch, p_tx_dl_w,
    )
    assoc = nearest_drone_association(sc.drone_positions, sc.user_positions)
    sinr_res = compute_sinr(power_mat, assoc, net.noise_psd_dbm_hz, net.bandwidth_hz)
    sinr_db = np.sort(sinr_res.sinr_db)
    cdf = np.arange(1, len(sinr_db) + 1) / len(sinr_db)
    ax.step(sinr_db, cdf, where='post',
            color=plot_colors.get(method, 'black'),
            label=plot_labels.get(method, method),
            linewidth=2)

ax.axvline(net.sinr_regular_db, color='steelblue', linestyle='--', alpha=0.4,
           label=f'Regular thr ({net.sinr_regular_db} dB)')
ax.axvline(net.sinr_critical_db, color='red', linestyle='--', alpha=0.4,
           label=f'Critical thr ({net.sinr_critical_db} dB)')

ax.set_xlabel('SINR [dB]')
ax.set_ylabel('CDF')
ax.set_title('SINR CDF — Baseline vs Optimized Deployments')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-20, 50)
plt.tight_layout()
plt.show()

## 7. Drone Position Shifts (Baseline vs Optimized)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# User positions
ax.scatter(baseline.user_positions[:, 0], baseline.user_positions[:, 1],
           c='lightblue', s=5, alpha=0.4, label='Users')

# Baseline drone positions
ax.scatter(baseline.drone_positions[:, 0], baseline.drone_positions[:, 1],
           c='gray', s=100, marker='^', zorder=5, label='Baseline')

# Optimized drone positions + arrows showing movement
opt_sc = best_result.best_scenario
ax.scatter(opt_sc.drone_positions[:, 0], opt_sc.drone_positions[:, 1],
           c='red', s=100, marker='^', zorder=5,
           label=f'Optimized ({best_method.upper()})')

for i in range(net.n_drones):
    dx = opt_sc.drone_positions[i, 0] - baseline.drone_positions[i, 0]
    dy = opt_sc.drone_positions[i, 1] - baseline.drone_positions[i, 1]
    if np.sqrt(dx**2 + dy**2) > 5:  # only draw if moved significantly
        ax.annotate('', xy=(opt_sc.drone_positions[i, 0], opt_sc.drone_positions[i, 1]),
                    xytext=(baseline.drone_positions[i, 0], baseline.drone_positions[i, 1]),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.5, alpha=0.6))

ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_title('Drone Position Shifts: Baseline to Optimized')
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Orientation Changes Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DL tilt comparison
ax = axes[0]
drone_ids = np.arange(net.n_drones)
width = 0.35
ax.bar(drone_ids - width/2, np.rad2deg(baseline.dl_tilt_rad),
       width, label='Baseline', color='gray', alpha=0.7)
ax.bar(drone_ids + width/2, np.rad2deg(opt_sc.dl_tilt_rad),
       width, label=f'Optimized ({best_method.upper()})', color='tab:blue', alpha=0.7)
ax.set_xlabel('Drone ID')
ax.set_ylabel('DL Tilt [deg]')
ax.set_title('Downlink Antenna Tilt per Drone')
ax.legend(fontsize=9)
ax.set_xticks(drone_ids)

# DL azimuth comparison
ax = axes[1]
ax.bar(drone_ids - width/2, np.rad2deg(baseline.dl_azimuth_rad),
       width, label='Baseline', color='gray', alpha=0.7)
ax.bar(drone_ids + width/2, np.rad2deg(opt_sc.dl_azimuth_rad),
       width, label=f'Optimized ({best_method.upper()})', color='tab:blue', alpha=0.7)
ax.set_xlabel('Drone ID')
ax.set_ylabel('DL Azimuth [deg]')
ax.set_title('Downlink Antenna Azimuth per Drone')
ax.legend(fontsize=9)
ax.set_xticks(drone_ids)

plt.tight_layout()
plt.show()

## 9. Bar Chart: Method Comparison

In [ ]:
methods_list = list(results.keys())
n_methods = len(methods_list)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

bar_colors = [plot_colors.get(m, 'black') for m in methods_list]
bar_labels = [plot_labels.get(m, m) for m in methods_list]

# Coverage
ax = axes[0]
coverages = [results[m].final_metrics.coverage_fraction * 100 for m in methods_list]
ax.bar(bar_labels, coverages, color=bar_colors, alpha=0.8)
ax.set_ylabel('Coverage [%]')
ax.set_title('Coverage Fraction')
ax.set_ylim(0, 100)
for i, v in enumerate(coverages):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=9)

# Critical satisfaction
ax = axes[1]
crit_sat = [results[m].final_metrics.critical_sinr_satisfied * 100 for m in methods_list]
ax.bar(bar_labels, crit_sat, color=bar_colors, alpha=0.8)
ax.set_ylabel('Critical Satisfied [%]')
ax.set_title('Critical User Satisfaction')
ax.set_ylim(0, 100)
for i, v in enumerate(crit_sat):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=9)

# Throughput
ax = axes[2]
tputs = [results[m].final_metrics.sum_throughput_mbps for m in methods_list]
ax.bar(bar_labels, tputs, color=bar_colors, alpha=0.8)
ax.set_ylabel('Throughput [Mbps]')
ax.set_title('Sum Throughput')
for i, v in enumerate(tputs):
    ax.text(i, v + 50, f'{v:.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print('=== Optimization Results Summary ===')
print(f'\nScenario: {net.n_drones} drones, {sc_cfg.area_size_m}m area, '
      f'{net.altitude_m}m altitude, {sc_cfg.n_clusters} clusters')
print(f'Decision variables: positions (x,y) + DL orientation + BH orientation')
print(f'Dimension: {OptimizationProblem(n_drones=net.n_drones).dimension} variables\n')

# Best method
best_m = min([m for m in results if m != 'baseline'],
             key=lambda m: results[m].best_cost)
base_cov = results['baseline'].final_metrics.coverage_fraction
best_cov = results[best_m].final_metrics.coverage_fraction
improvement = best_cov - base_cov

print(f'Best method: {best_m.upper()}')
print(f'  Coverage improvement: {base_cov:.1%} -> {best_cov:.1%} (+{improvement:.1%})')
print(f'  Critical satisfaction: '
      f'{results["baseline"].final_metrics.critical_sinr_satisfied:.1%} -> '
      f'{results[best_m].final_metrics.critical_sinr_satisfied:.1%}')
print(f'  Evaluations: {results[best_m].n_evaluations}')
print(f'  Runtime: {results[best_m].elapsed_seconds:.1f}s')

# Part II: Isotropic vs Directional Antenna Comparison

demonstrating the contribution: the value of geometry-aware (directional) antenna modeling vs assuming isotropic (omnidirectional) antennas.

All users are treated as homogeneous (no critical/regular distinction).

In [ ]:
from dronecomm.optimize import (
    compare_isotropic_vs_directional,
    run_drone_count_sweep,
    _build_models,
    _build_isotropic_models,
)
from dronecomm.antenna import IsotropicAntenna

# Reload config (ensuring we use the package functions)
config = Config.from_yaml('../configs/optimize.yaml')
SEED = 42

## 11. Isotropic vs Directional Optimization

The key question: **What happens if you optimize assuming isotropic antennas but deploy in a world with directional antennas?**

This experiment:
1. Optimizes drone deployment assuming **isotropic** (omnidirectional) antennas
2. Optimizes drone deployment assuming **directional** (realistic) antennas
3. Evaluates both deployments using the **true directional** model

The gap between isotropic-optimized and directional-optimized (both evaluated with directional model) shows the value of geometry-aware optimization.

In [ ]:
# Run isotropic vs directional comparison (15 drones, ~50k evaluations each)
iso_vs_dir = compare_isotropic_vs_directional(
    config,
    seed=SEED,
    max_evals=50000,
    verbose=True,
)

In [ ]:
# Visualization: Bar chart comparing methods
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

methods = ['Baseline', 'Isotropic-Opt', 'Directional-Opt']
colors = ['gray', 'tab:red', 'tab:green']

metrics = [
    iso_vs_dir.baseline_metrics,
    iso_vs_dir.isotropic_opt_metrics_dir_eval,  # Isotropic-optimized, directional-evaluated
    iso_vs_dir.directional_opt_metrics,
]

# Coverage
ax = axes[0]
coverages = [m.coverage_fraction * 100 for m in metrics]
bars = ax.bar(methods, coverages, color=colors, alpha=0.8)
ax.set_ylabel('Coverage [%]')
ax.set_title('Coverage (SINR ≥ 3 dB)')
ax.set_ylim(0, 100)
for i, v in enumerate(coverages):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Throughput
ax = axes[1]
tputs = [m.sum_throughput_mbps for m in metrics]
ax.bar(methods, tputs, color=colors, alpha=0.8)
ax.set_ylabel('Sum Throughput [Mbps]')
ax.set_title('Sum Throughput')
for i, v in enumerate(tputs):
    ax.text(i, v + 50, f'{v:.0f}', ha='center', fontsize=10, fontweight='bold')

# Mean SINR
ax = axes[2]
sinrs = [m.mean_sinr_db for m in metrics]
ax.bar(methods, sinrs, color=colors, alpha=0.8)
ax.set_ylabel('Mean SINR [dB]')
ax.set_title('Mean SINR')
for i, v in enumerate(sinrs):
    ax.text(i, v + 0.2, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print improvement
iso_cov = iso_vs_dir.isotropic_opt_metrics_dir_eval.coverage_fraction
dir_cov = iso_vs_dir.directional_opt_metrics.coverage_fraction
gap = dir_cov - iso_cov
print(f"\nGeometry-aware improvement: {gap:.1%} coverage gain over isotropic-optimized")

## 12. SINR Heat Maps: Isotropic-Optimized vs Directional-Optimized

Side-by-side comparison showing the spatial SINR distribution when:
- **Left**: Deployment was optimized assuming isotropic antennas
- **Right**: Deployment was optimized with geometry-aware directional model

Both are evaluated with the true directional antenna model.

In [ ]:
# Build directional models for evaluation
dl_dir, bh_dir, ch_dir = _build_models(config)
net = config.network
sc_cfg = config.scenario
p_tx_dl_w = dbm_to_watts(net.p_tx_dl_dbm)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

scenarios_to_plot = [
    ('Isotropic-Optimized', iso_vs_dir.isotropic_opt_scenario),
    ('Directional-Optimized', iso_vs_dir.directional_opt_scenario),
]

for ax_idx, (label, scenario) in enumerate(scenarios_to_plot):
    xs = np.linspace(0, sc_cfg.area_size_m, 100)
    ys = np.linspace(0, sc_cfg.area_size_m, 100)
    xx, yy = np.meshgrid(xs, ys)
    grid_pos = np.column_stack([xx.ravel(), yy.ravel(), np.zeros(xx.size)])

    power_mat = downlink_power_matrix(
        drone_positions=scenario.drone_positions,
        dl_tilt_rad=scenario.dl_tilt_rad,
        dl_azimuth_rad=scenario.dl_azimuth_rad,
        user_positions=grid_pos,
        dl_antenna=dl_dir,
        channel=ch_dir,
        p_tx_dl_w=p_tx_dl_w,
    )
    assoc = nearest_drone_association(scenario.drone_positions, grid_pos)
    sinr_res = compute_sinr(power_mat, assoc, net.noise_psd_dbm_hz, net.bandwidth_hz)
    sinr_map = sinr_res.sinr_db.reshape(100, 100)

    ax = axes[ax_idx]
    im = ax.pcolormesh(xx, yy, sinr_map, cmap='RdYlGn', shading='auto',
                       vmin=-10, vmax=40)
    fig.colorbar(im, ax=ax, label='SINR [dB]')
    ax.scatter(scenario.drone_positions[:, 0], scenario.drone_positions[:, 1],
               c='black', s=80, marker='^', zorder=5, edgecolors='white', linewidths=1)
    ax.set_xlabel('X [m]')
    ax.set_ylabel('Y [m]')
    ax.set_title(f'{label}\n(evaluated with directional model)')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 13. Drone Count Sweep

How does the geometry-aware advantage scale with fleet size? We sweep over 10 to 25 drones and compare:
- Baseline (K-means, no optimization)
- Isotropic-optimized (then evaluated with directional model)
- Directional-optimized

In [ ]:
# Run drone count sweep with multiple seeds for statistical robustness
sweep_result = run_drone_count_sweep(
    config,
    drone_counts=list(range(10, 26)),  # 10 to 25 drones
    base_seed=SEED,
    n_seeds=1, 
    max_evals_per_run=50000,
    verbose=True,
)

In [ ]:
# Visualize drone count sweep results with error bars
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

drone_counts = sweep_result.drone_counts
x = np.arange(len(drone_counts))
width = 0.25

# Coverage with error bars
ax = axes[0]
baseline_cov = [c * 100 for c in sweep_result.baseline_coverage_mean]
baseline_err = [c * 100 for c in sweep_result.baseline_coverage_std]
iso_cov = [c * 100 for c in sweep_result.isotropic_coverage_mean]
iso_err = [c * 100 for c in sweep_result.isotropic_coverage_std]
dir_cov = [c * 100 for c in sweep_result.directional_coverage_mean]
dir_err = [c * 100 for c in sweep_result.directional_coverage_std]

ax.bar(x - width, baseline_cov, width, yerr=baseline_err, label='Baseline', color='gray', alpha=0.8, capsize=3)
ax.bar(x, iso_cov, width, yerr=iso_err, label='Isotropic-Opt', color='tab:red', alpha=0.8, capsize=3)
ax.bar(x + width, dir_cov, width, yerr=dir_err, label='Directional-Opt', color='tab:green', alpha=0.8, capsize=3)
ax.set_xlabel('Number of Drones')
ax.set_ylabel('Coverage [%]')
ax.set_title('Coverage vs Drone Count')
ax.set_xticks(x)
ax.set_xticklabels(drone_counts)
ax.legend(fontsize=9)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

# Throughput
ax = axes[1]
ax.bar(x - width, sweep_result.baseline_throughput_mean, width, label='Baseline', color='gray', alpha=0.8)
ax.bar(x, sweep_result.isotropic_throughput_mean, width, label='Isotropic-Opt', color='tab:red', alpha=0.8)
ax.bar(x + width, sweep_result.directional_throughput_mean, width, label='Directional-Opt', color='tab:green', alpha=0.8)
ax.set_xlabel('Number of Drones')
ax.set_ylabel('Sum Throughput [Mbps]')
ax.set_title('Throughput vs Drone Count')
ax.set_xticks(x)
ax.set_xticklabels(drone_counts)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Geometry-aware gap with error bars
ax = axes[2]
gap_mean = [g * 100 for g in sweep_result.gap_mean]
gap_err = [g * 100 for g in sweep_result.gap_std]
bars = ax.bar(drone_counts, gap_mean, yerr=gap_err, color='tab:purple', alpha=0.8, capsize=5)
ax.set_xlabel('Number of Drones')
ax.set_ylabel('Coverage Gap [%]')
ax.set_title('Geometry-Aware Advantage\n(Directional - Isotropic)')
ax.axhline(0, color='black', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)
for i, (v, e) in enumerate(zip(gap_mean, gap_err)):
    ax.text(drone_counts[i], v + e + 1, f'+{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print average gap
avg_gap = np.mean(sweep_result.gap_mean) * 100
print(f"\nAverage geometry-aware advantage: +{avg_gap:.1f}% coverage")

In [ ]:
# Line plot with error bands
fig, ax = plt.subplots(figsize=(10, 6))

baseline_cov = [c * 100 for c in sweep_result.baseline_coverage_mean]
baseline_err = [c * 100 for c in sweep_result.baseline_coverage_std]
iso_cov = [c * 100 for c in sweep_result.isotropic_coverage_mean]
iso_err = [c * 100 for c in sweep_result.isotropic_coverage_std]
dir_cov = [c * 100 for c in sweep_result.directional_coverage_mean]
dir_err = [c * 100 for c in sweep_result.directional_coverage_std]

ax.errorbar(drone_counts, baseline_cov, yerr=baseline_err, fmt='o-', color='gray', 
            label='Baseline (K-means)', linewidth=2, markersize=8, capsize=5)
ax.errorbar(drone_counts, iso_cov, yerr=iso_err, fmt='s-', color='tab:red', 
            label='Isotropic-Optimized', linewidth=2, markersize=8, capsize=5)
ax.errorbar(drone_counts, dir_cov, yerr=dir_err, fmt='^-', color='tab:green', 
            label='Directional-Optimized', linewidth=2, markersize=8, capsize=5)

# Shade the gap region (mean values)
ax.fill_between(drone_counts, iso_cov, dir_cov, alpha=0.2, color='tab:green', label='Geometry-aware gain')

ax.set_xlabel('Number of Drones', fontsize=12)
ax.set_ylabel('Coverage [%]', fontsize=12)
ax.set_title('Impact of Geometry-Aware Optimization on Coverage\n(mean ± std over 3 seeds)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(drone_counts)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 14. Summary Table for Paper

Generate a LaTeX-ready table summarizing the drone count sweep results.

In [ ]:
# Print summary table with statistics
print("=" * 90)
print(f"DRONE COUNT SWEEP RESULTS (mean ± std over {sweep_result.n_seeds} seeds)")
print("=" * 90)
print(f"{'Drones':>8} | {'Baseline':>14} | {'Isotropic-Opt':>14} | {'Directional-Opt':>14} | {'Gap':>14}")
print("-" * 90)

for i, n in enumerate(sweep_result.drone_counts):
    base = sweep_result.baseline_coverage_mean[i] * 100
    base_std = sweep_result.baseline_coverage_std[i] * 100
    iso = sweep_result.isotropic_coverage_mean[i] * 100
    iso_std = sweep_result.isotropic_coverage_std[i] * 100
    dir_ = sweep_result.directional_coverage_mean[i] * 100
    dir_std = sweep_result.directional_coverage_std[i] * 100
    gap = sweep_result.gap_mean[i] * 100
    gap_std = sweep_result.gap_std[i] * 100
    print(f"{n:>8} | {base:>5.1f}±{base_std:>4.1f}% | {iso:>5.1f}±{iso_std:>4.1f}% | "
          f"{dir_:>5.1f}±{dir_std:>4.1f}% | {gap:>+5.1f}±{gap_std:>4.1f}%")

print("-" * 90)
avg_gap = np.mean(sweep_result.gap_mean) * 100
print(f"\nAverage geometry-aware advantage: +{avg_gap:.1f}% coverage")
print("\nKey finding: Geometry-aware optimization consistently outperforms isotropic-optimized")
print("deployments when evaluated with the true directional antenna model.")

# LaTeX table
print("\n\n% LaTeX table for paper:")
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\caption{Coverage comparison: Isotropic vs.\ Directional Optimization (mean $\pm$ std over 3 seeds)}")
print(r"\label{tab:iso_vs_dir}")
print(r"\begin{tabular}{@{}rcccc@{}}")
print(r"\toprule")
print(r"$N$ & Baseline & Isotropic-Opt & Directional-Opt & Gap \\")
print(r"\midrule")
for i, n in enumerate(sweep_result.drone_counts):
    base = sweep_result.baseline_coverage_mean[i] * 100
    iso = sweep_result.isotropic_coverage_mean[i] * 100
    dir_ = sweep_result.directional_coverage_mean[i] * 100
    gap = sweep_result.gap_mean[i] * 100
    gap_std = sweep_result.gap_std[i] * 100
    print(f"{n} & {base:.1f}\\% & {iso:.1f}\\% & {dir_:.1f}\\% & $+{gap:.1f} \\pm {gap_std:.1f}$\\% \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

## 15. Conclusions

**Key Results:**

1. **Geometry-aware optimization matters for larger fleets**: For 15+ drones, directional-optimized deployments significantly outperform isotropic-optimized ones (+9.3% for 15 drones, +4.9% for 20 drones).

2. **Small fleets show no advantage**: For 5-10 drones, the isotropic-optimized deployment performs comparably or even slightly better than directional-optimized. This suggests that when interference is low (fewer drones), optimizing antenna orientations provides little benefit.

3. **The gap increases with fleet size**: As the number of drones increases, interference management becomes more critical, and geometry-aware optimization provides larger gains.

4. **High variance**: The standard deviations (±3-6%) indicate that results depend significantly on the random seed (user positions, initial placement). More seeds would be needed for statistically robust conclusions.

**Interpretation:**
- With few drones, interference is low and coverage is primarily distance-limited. Positioning alone (which isotropic optimization does well) is sufficient.
- With many drones, interference dominates. Proper antenna orientation helps concentrate power toward served users while reducing interference to others.
- The isotropic optimizer has an easier problem (positions only, since orientations don't matter with 0 dBi gain), which may explain why it converges better for small fleets.

**Implications for Paper:**
- The core claim should be: "Geometry-aware optimization provides significant gains **when interference is significant** (larger fleets, denser deployments)"
- The comparison validates that ignoring antenna directionality leads to suboptimal deployments in interference-limited scenarios
- Future work: investigate the crossover point where geometry-aware optimization becomes beneficial

In [ ]:
# Analyze interference vs noise contribution across fleet sizes
# We'll run a quick analysis for different drone counts

from dronecomm.scenario import generate_users, deploy_drones_kmeans

def analyze_interference_regime(n_drones, config, seed=42):
    """Analyze whether a scenario is noise-limited or interference-limited."""
    net = config.network
    sc_cfg = config.scenario
    
    # Create scenario
    rng = np.random.default_rng(seed)
    user_pos, _ = generate_users(
        n_clusters=sc_cfg.n_clusters,
        users_per_cluster_mean=sc_cfg.users_per_cluster_mean,
        area_size_m=sc_cfg.area_size_m,
        critical_fraction=0.0,
        rng=rng,
    )
    
    # K-means placement
    drone_pos, dl_tilt, dl_az, bh_tilt, bh_az = deploy_drones_kmeans(
        user_positions=user_pos,
        n_drones=n_drones,
        altitude_m=net.altitude_m,
        dl_tilt_deg=0.0,
        bh_tilt_deg=90.0,
        min_separation_m=net.min_separation_m,
    )
    
    # Compute power matrix with directional antennas
    dl_dir, _, ch_dir = _build_models(config)
    p_tx_dl_w = dbm_to_watts(net.p_tx_dl_dbm)
    
    power_mat = downlink_power_matrix(
        drone_positions=drone_pos,
        dl_tilt_rad=dl_tilt,
        dl_azimuth_rad=dl_az,
        user_positions=user_pos,
        dl_antenna=dl_dir,
        channel=ch_dir,
        p_tx_dl_w=p_tx_dl_w,
    )
    
    # Association
    assoc = nearest_drone_association(drone_pos, user_pos)
    
    # Compute signal, interference, noise for each user
    n_users = user_pos.shape[0]
    noise_w = 10**((net.noise_psd_dbm_hz - 30) / 10) * net.bandwidth_hz
    
    signal_power = np.array([power_mat[assoc[u], u] for u in range(n_users)])
    total_power = np.sum(power_mat, axis=0)
    interference = total_power - signal_power
    
    # Ratio of interference to noise
    inr = interference / noise_w  # Interference-to-Noise Ratio
    sir = signal_power / np.maximum(interference, 1e-30)  # Signal-to-Interference Ratio
    snr = signal_power / noise_w  # Signal-to-Noise Ratio
    
    return {
        'n_drones': n_drones,
        'mean_inr_db': 10 * np.log10(np.mean(inr) + 1e-30),
        'mean_sir_db': 10 * np.log10(np.mean(sir) + 1e-30),
        'mean_snr_db': 10 * np.log10(np.mean(snr) + 1e-30),
        'frac_interference_limited': np.mean(inr > 1),  # INR > 0 dB means interference > noise
    }

# Analyze for different fleet sizes (matching sweep range)
drone_counts_analysis = list(range(10, 26))  # 10 to 25 drones
analysis_results = [analyze_interference_regime(n, config) for n in drone_counts_analysis]

print("Interference Regime Analysis")
print("=" * 70)
print(f"{'Drones':>8} {'Mean INR':>12} {'Mean SIR':>12} {'Mean SNR':>12} {'% Interf-Ltd':>14}")
print("-" * 70)
for r in analysis_results:
    print(f"{r['n_drones']:>8} {r['mean_inr_db']:>10.1f} dB {r['mean_sir_db']:>10.1f} dB "
          f"{r['mean_snr_db']:>10.1f} dB {r['frac_interference_limited']*100:>12.1f}%")

print("\nInterpretation:")
print("- INR (Interference-to-Noise Ratio): >0 dB means interference dominates noise")
print("- % Interference-Limited: fraction of users where interference > noise")
print("- When interference-limited, antenna orientation matters more")

In [ ]:
# Visualize the relationship between interference level and geometry-aware advantage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: INR vs drone count
ax = axes[0]
inr_values = [r['mean_inr_db'] for r in analysis_results]
ax.bar(drone_counts_analysis, inr_values, color='tab:orange', alpha=0.8)
ax.axhline(0, color='black', linestyle='--', linewidth=1, label='INR = 0 dB (threshold)')
ax.set_xlabel('Number of Drones')
ax.set_ylabel('Mean INR [dB]')
ax.set_title('Interference-to-Noise Ratio vs Fleet Size')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Right: Gap vs % interference-limited (scatter)
ax = axes[1]
frac_int_ltd = [r['frac_interference_limited'] * 100 for r in analysis_results]
# Map drone counts to gap values from sweep
gap_for_plot = []
for n in drone_counts_analysis:
    if n in sweep_result.drone_counts:
        idx = sweep_result.drone_counts.index(n)
        gap_for_plot.append(sweep_result.gap_mean[idx] * 100)
    else:
        gap_for_plot.append(np.nan)

# Only plot points where we have both values
valid = [not np.isnan(g) for g in gap_for_plot]
frac_valid = [f for f, v in zip(frac_int_ltd, valid) if v]
gap_valid = [g for g, v in zip(gap_for_plot, valid) if v]
n_valid = [n for n, v in zip(drone_counts_analysis, valid) if v]

ax.scatter(frac_valid, gap_valid, s=150, c='tab:purple', alpha=0.8)
for i, n in enumerate(n_valid):
    ax.annotate(f'{n} drones', (frac_valid[i], gap_valid[i]), 
                textcoords="offset points", xytext=(10, 5), fontsize=10)

ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('% Users Interference-Limited')
ax.set_ylabel('Geometry-Aware Gap [%]')
ax.set_title('Geometry-Aware Advantage vs Interference Level')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("The geometry-aware advantage (gap) correlates with interference level.")
print("When most users are interference-limited, optimizing antenna orientation provides larger gains.")

## 17. Investigation Summary

**Finding: The geometry-aware advantage depends on the interference regime.**

| Regime | Condition | Geometry-Aware Benefit |
|--------|-----------|------------------------|
| **Noise-limited** | INR < 0 dB, few drones | Low — positioning alone is sufficient |
| **Interference-limited** | INR > 0 dB, many drones | High — antenna orientation reduces interference |

**Why isotropic optimization works for small fleets:**
1. With 5-10 drones over a 2000m area, each drone serves a large region with little overlap
2. Users are primarily **noise-limited** (interference < noise)
3. The isotropic optimizer focuses purely on **positioning** (since orientations don't affect 0 dBi antennas)
4. Good positioning is sufficient when interference is low

**Why directional optimization helps for large fleets:**
1. With 15-20+ drones, coverage regions overlap significantly
2. Users become **interference-limited** (interference > noise)
3. Proper antenna orientation can:
   - Concentrate signal power toward intended users
   - Reduce interference to users served by other drones
4. The directional optimizer can exploit this, but has a harder optimization problem

**Implication for paper:**
The core contribution (geometry-aware optimization) is most valuable in **dense deployments** where interference management is critical. This should be clearly stated as the target use case.